In [1]:
# Import Libraries
import numpy as np
import torch

In [ ]:
# Goal of this notebook is to learn the very basics of pytorch:
# What a tensor is in pytorch, 

"""
KEY FUNCTIONS
.storage() --> The actual flat block of memory — just a 1D array of numbers
.shape / .size()  --> The dimensions (like MATLAB's size(A))
.stride()  --> How many elements to skip in storage to move one step along each dimension
.dtype  --> Data type (float32, int64, etc.)
.device  --> Where it lives — cpu or cuda:0 (GPU)
.requires_grad  --> Whether to track operations for backprop
.grad_fn  --> The function that created this tensor (the autograd graph link)
.grad  --> Where gradients accumulate after .backward()
.contiguous() --> .contiguous() copies to a fresh sequential layout
""" 


"\nKEY FUNCTIONS\n.storage() --> The actual flat block of memory — just a 1D array of numbers\n.shape / .size()  --> The dimensions (like MATLAB's size(A))\n.stride()  --> How many elements to skip in storage to move one step along each dimension\n.dtype  --> Data type (float32, int64, etc.)\n.device  --> Where it lives — cpu or cuda:0 (GPU)\n.requires_grad  --> Whether to track operations for backprop\n.grad_fn  --> The function that created this tensor (the autograd graph link)\n.grad  --> Where gradients accumulate after .backward()\n"

In [28]:
# CREATING TENSORS
# From data (like MATLAB literals)
a = torch.tensor([1.0, 2.0, 3.0])
B = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

# Allocation (like zeros(), ones(), rand() in MATLAB)
torch.zeros(3, 4)
torch.ones(2, 3)
torch.randn(3, 4)        # normal distribution
torch.rand(3, 4)         # uniform [0, 1)
print( "torch.arange(0, 10, 2)", torch.arange(0, 10, 2))   # like MATLAB 0:2:8

torch.linspace(0, 1, 5)  # like MATLAB linspace

# From NumPy (shared memory — no copy!)
import numpy as np
n = np.array([1.0, 2.0])
t = torch.from_numpy(n)  # mutating t mutates n

torch.arange(0, 10, 2) tensor([0, 2, 4, 6, 8])


In [ ]:
# Create a tensor
X = torch.arange(24)
print("X pre-reshape: ", X)
print(X.stride())

print()
X = X.reshape(2,3,4)
# print(X.storage())
print(X)
print( "X stride: ", X.stride())
print("X shape: ", X.shape)
print(X.size())

Y = X.T

print( "Y stride: ", Y.stride() )
#print( " Y storage: ", Y.storage() ) # Same memory
print( " Y contiguous: ", Y.is_contiguous() )


tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])
(1,)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
X stride:  (12, 4, 1)
X shape:  torch.Size([2, 3, 4])
torch.Size([2, 3, 4])
Y stride:  (1, 4, 12)
 Y contiguous:  False


.view() requires contiguous memory. .reshape() will silently copy if needed. When you're doing thousands of reshapes in a training loop, knowing when copies happen is the difference between fast and OOM.

In [21]:
# Reshape tensors
X = torch.randn(2, 3, 4)  # shape (2, 3, 4)
print( "Original X", X)
print( "Dim", X.shape)

# Reshape — reinterpret shape (may copy if non-contiguous)
X = X.reshape(6, 4)
print( "Reshaped X", X)
print( "Dim", X.shape)
X = X.reshape(-1)          # flatten, like MATLAB's A(:)
print( "Flattened X", X)
print( "Dim", X.shape)
X = X.reshape(2,3,4)

# View -- This explicitly does not copy - fails if non-contiguous
print( "view as 2x12:", X.view(2,12) )

print( "X transposed (0,2) ", X.transpose( 0,2 )) 
print( "X transpose shape ", X.transpose( 0,2 ).shape )

# Permute dimensions
print( "X Permuted (2,0,1) ", X.permute( 2,0,1 )) 
print( "X Permuted shape ", X.permute( 2,0,1 ).shape )


Original X tensor([[[-0.9343,  0.4691, -0.4775,  0.5228],
         [ 0.2213,  1.9180,  0.1765,  0.3637],
         [-1.3053,  1.0166, -0.0212,  0.1850]],

        [[-1.6099, -1.1930, -0.7984, -2.4513],
         [ 0.1004, -0.0319, -0.0763,  1.4178],
         [ 0.5628, -0.1182,  0.8852, -0.0119]]])
Dim torch.Size([2, 3, 4])
Reshaped X tensor([[-0.9343,  0.4691, -0.4775,  0.5228],
        [ 0.2213,  1.9180,  0.1765,  0.3637],
        [-1.3053,  1.0166, -0.0212,  0.1850],
        [-1.6099, -1.1930, -0.7984, -2.4513],
        [ 0.1004, -0.0319, -0.0763,  1.4178],
        [ 0.5628, -0.1182,  0.8852, -0.0119]])
Dim torch.Size([6, 4])
Flattened X tensor([-0.9343,  0.4691, -0.4775,  0.5228,  0.2213,  1.9180,  0.1765,  0.3637,
        -1.3053,  1.0166, -0.0212,  0.1850, -1.6099, -1.1930, -0.7984, -2.4513,
         0.1004, -0.0319, -0.0763,  1.4178,  0.5628, -0.1182,  0.8852, -0.0119])
Dim torch.Size([24])
view as 2x12: tensor([[-0.9343,  0.4691, -0.4775,  0.5228,  0.2213,  1.9180,  0.1765,  0.363

In [24]:
# Squeeze / Un-squeeze
a = torch.randn(3)
print( "A", a )
print( "Shape: ", a.shape )
print()

print( "a Unsqueezed(0): ", a.unsqueeze(0) )         # shape (1, 3) — like MATLAB row vector
print( a.unsqueeze(0) )
print(a.unsqueeze(0).shape)         # shape (3, 1) — like MATLAB column vector

print()
print( "a Unsqueezed(1): ", a.unsqueeze(1) )         # shape (1, 3) — like MATLAB row vector
print( a.unsqueeze(1) )
print(a.unsqueeze(1).shape)

A tensor([-1.2340,  0.0262, -0.8754])
Shape:  torch.Size([3])

a Unsqueezed(0):  tensor([[-1.2340,  0.0262, -0.8754]])
tensor([[-1.2340,  0.0262, -0.8754]])
torch.Size([1, 3])

a Unsqueezed(1):  tensor([[-1.2340],
        [ 0.0262],
        [-0.8754]])
tensor([[-1.2340],
        [ 0.0262],
        [-0.8754]])
torch.Size([3, 1])


In [29]:
# Expand / repeat — broadcast without copying
a = torch.randn(1, 4)
print( a )

print( a.expand(3, 4) )         # shape (3, 4), no new memory (stride 0 trick)
print(a)

tensor([[-0.8138, -0.9283,  0.3270,  0.7842]])
tensor([[-0.8138, -0.9283,  0.3270,  0.7842],
        [-0.8138, -0.9283,  0.3270,  0.7842],
        [-0.8138, -0.9283,  0.3270,  0.7842]])
tensor([[-0.8138, -0.9283,  0.3270,  0.7842]])


In [26]:
# Indexing

X = torch.randn(4, 5)
print(X)
print()


print("X[0] ", X[0] )         # first row
print() 


print( "X[:, 0] ", X[:, 0] )       # first column
print() 

print( "X[1:3, 2:4] ", X[1:3, 2:4])   # slice
print() 

print( "X[X > 0] ", X[X > 0] )
X[X > 0]      # boolean mask (like MATLAB logical indexing)
print() 

# Advanced indexing (gathers specific elements)
print( "idx = torch.tensor([0, 2, 3])")
idx = torch.tensor([0, 2, 3])
print( idx)
print() 

print( "X[idx]", X[idx] )        # rows 0, 2, 3


tensor([[-0.3660, -1.1137,  0.5784,  1.1633, -0.7807],
        [ 1.0909,  0.9701,  0.4171, -0.6643, -0.4574],
        [-0.3423, -0.7680, -1.6128, -1.7856, -0.5627],
        [ 0.2647,  2.2579, -0.6869, -0.7859, -0.9454]])

X[0]  tensor([-0.3660, -1.1137,  0.5784,  1.1633, -0.7807])

X[:, 0]  tensor([-0.3660,  1.0909, -0.3423,  0.2647])

X[1:3, 2:4]  tensor([[ 0.4171, -0.6643],
        [-1.6128, -1.7856]])

X[X > 0]  tensor([0.5784, 1.1633, 1.0909, 0.9701, 0.4171, 0.2647, 2.2579])

idx = torch.tensor([0, 2, 3])
tensor([0, 2, 3])

X[idx] tensor([[-0.3660, -1.1137,  0.5784,  1.1633, -0.7807],
        [-0.3423, -0.7680, -1.6128, -1.7856, -0.5627],
        [ 0.2647,  2.2579, -0.6869, -0.7859, -0.9454]])


In [27]:
# Element-wise manipulations (like MATLAB .* ./ .^)
A = torch.randn(2,3)
B = torch.randn(2,3)
print( "A + B: ", A + B )
print("A * B ", A * B )         # element-wise multiply
print( "A ** 2 ", A ** 2 )         # element-wise power
print( "torch.exp(A)", torch.exp(A) )
print( "torch.log(A) ", torch.log(A))
print( "torch.relu(A)", torch.relu(A))  # max(0, x) element-wise

A + B:  tensor([[ 1.5818,  0.9090, -3.5414],
        [ 0.5517,  3.1175, -0.3529]])
A * B  tensor([[0.6193, 0.1191, 2.6048],
        [0.0328, 1.7553, 0.0272]])
A ** 2  tensor([[7.5661e-01, 5.6302e-01, 1.0864e+00],
        [4.5924e-03, 5.6643e+00, 5.7041e-02]])
torch.exp(A) tensor([[ 2.3865,  2.1177,  0.3526],
        [ 1.0701, 10.8047,  0.7875]])
torch.log(A)  tensor([[-0.1395, -0.2872,     nan],
        [-2.6917,  0.8671,     nan]])
torch.relu(A) tensor([[0.8698, 0.7503, 0.0000],
        [0.0678, 2.3800, 0.0000]])


In [31]:
# Matrix multiply (like MATLAB A * B or A @ B)
A = torch.randn(2,3)
B = torch.randn(3,2)
A @ B                    # preferred syntax
torch.matmul(A, B)      # same thing
torch.mm(A, B)          # strictly 2D only

tensor([[-1.5178,  1.4415],
        [-1.1554,  1.8093]])

In [37]:
# Batched matmul — this is critical for attention
# (batch, n, m) @ (batch, m, p) → (batch, n, p)
A = torch.randn(2,3,1)
B = torch.randn(2,1,3)

torch.bmm(A, B)

tensor([[[ 1.0863,  0.3643,  0.0700],
         [ 0.7271,  0.2438,  0.0469],
         [ 0.4052,  0.1359,  0.0261]],

        [[ 1.0413,  0.0897,  0.5073],
         [-0.2123, -0.0183, -0.1034],
         [ 0.9577,  0.0825,  0.4666]]])

In [38]:
# Einstein summation — the power tool
torch.einsum('ij,jk->ik', A, B)       # matmul
torch.einsum('bij,bkj->bik', Q, K)    # batched dot product (attention)


RuntimeError: einsum(): the number of subscripts in the equation (2) does not match the number of dimensions (3) for operand 0 and no ellipsis was given

In [ ]:
# Reductions (like MATLAB sum, mean, max along a dim)
X.sum(dim=0)       # sum along rows → shape (5,)
X.mean(dim=1)      # mean along columns → shape (4,)
X.max(dim=1)       # returns (values, indices) tuple